# 🔄 Модуль 2. Урок 6: Паттерны продуктовой аналитики: Воронки и Когорты на SQL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_06_sql_patterns_retention_funnels.ipynb)

---

## 1. Помесячный когортный анализ Retention Rate на чистом SQL

Когортный анализ отвечает на ключевой вопрос: *«Удерживаем ли мы клиентов, привлеченных в разные месяцы, или они разово покупают и исчезают?»*


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

query = """
WITH cohorts AS (
    SELECT 
        customer_id,
        MIN(SUBSTR(order_date, 1, 7)) AS first_order_month
    FROM orders
    WHERE status = 'Completed'
    GROUP BY customer_id
),
activities AS (
    SELECT 
        o.customer_id,
        c.first_order_month,
        SUBSTR(o.order_date, 1, 7) AS order_month
    FROM orders o
    JOIN cohorts c ON o.customer_id = c.customer_id
    WHERE o.status = 'Completed'
)
SELECT 
    first_order_month AS cohort,
    COUNT(DISTINCT customer_id) AS cohort_size,
    COUNT(DISTINCT CASE WHEN order_month = first_order_month THEN customer_id END) AS m0_active,
    COUNT(DISTINCT CASE WHEN order_month > first_order_month THEN customer_id END) AS repeat_buyers,
    ROUND(COUNT(DISTINCT CASE WHEN order_month > first_order_month THEN customer_id END) * 100.0 / 
          COUNT(DISTINCT customer_id), 1) AS repeat_buyer_pct
FROM activities
GROUP BY first_order_month
ORDER BY first_order_month
LIMIT 6;
"""

print("КОГОРТНАЯ МАТРИЦА УДЕРЖАНИЯ:")
print(pd.read_sql(query, conn))
